# 글로벌 공개 ATS 30개 기술기업 파일럿 및 Title -> 직무 검출

Greenhouse, Lever, Ashby의 **공개 채용 보드 GET API**를 이용해 현재 게시 중인 전체 JD를 수집하고, 개발·데이터 직군 비율과 기술 스택 추출 가능성을 검증합니다.

## API 키가 필요한가?

**필요 없습니다.** 이 노트북이 사용하는 공개 채용공고 조회 GET에는 인증키가 없습니다.

- Greenhouse: 공개 Job Board GET은 인증 불필요. 지원서 제출 POST만 인증 필요
- Lever: 공개 Postings API의 JSON 조회 사용
- Ashby: Public Job Postings API 사용

따라서 `.env`, API 키 발급, 로그인 셀이 없습니다.

> 공개 조회 가능 여부와 원문 재배포 권리는 같은 개념이 아닙니다. 파일럿은 내부 분석용으로 수행하고, 외부 서비스에서는 원문 복제보다 분석 결과와 원문 링크 제공을 우선합니다.

## 0. 준비

필요 패키지는 `requests`, `pandas`입니다. 현재 환경에 없다면 아래 주석을 해제해 한 번만 설치하세요.

In [1]:
# %pip install requests pandas

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from html import unescape
from pathlib import Path
import json
import re
import time

import pandas as pd
import requests
from IPython.display import display

REQUEST_TIMEOUT = 30
MAX_WORKERS = 6
SAVE_RESULTS = False  # 저장하려면 마지막 실행 전에 True로 변경
USER_AGENT = 'DevCompass-public-ATS-pilot/0.1 (academic data validation)'
COLLECTED_AT = datetime.now(timezone.utc).isoformat()

print('수집 시각(UTC):', COLLECTED_AT)
print('API 키: 필요 없음')

수집 시각(UTC): 2026-08-11T13:48:14.537617+00:00
API 키: 필요 없음


## 1. 회사·ATS 레지스트리

아래 30개 보드는 2026-08-10에 공개 endpoint가 HTTP 200을 반환하고 공고가 1건 이상 존재함을 확인한 표본입니다. 보드 이전이나 회사의 ATS 교체로 나중에 실패할 수 있으며, 그 실패 자체도 운영상 중요한 관측값입니다.

In [2]:
COMPANIES = [
    # Greenhouse (12)
    {'company': 'Stripe', 'ats': 'greenhouse', 'board_slug': 'stripe'},
    {'company': 'Cloudflare', 'ats': 'greenhouse', 'board_slug': 'cloudflare'},
    {'company': 'Datadog', 'ats': 'greenhouse', 'board_slug': 'datadog'},
    {'company': 'Discord', 'ats': 'greenhouse', 'board_slug': 'discord'},
    {'company': 'Figma', 'ats': 'greenhouse', 'board_slug': 'figma'},
    {'company': 'Cockroach Labs', 'ats': 'greenhouse', 'board_slug': 'cockroachlabs'},
    {'company': 'MongoDB', 'ats': 'greenhouse', 'board_slug': 'mongodb'},
    {'company': 'Lyft', 'ats': 'greenhouse', 'board_slug': 'lyft'},
    {'company': 'Roblox', 'ats': 'greenhouse', 'board_slug': 'roblox'},
    {'company': 'Airbnb', 'ats': 'greenhouse', 'board_slug': 'airbnb'},
    {'company': 'Anthropic', 'ats': 'greenhouse', 'board_slug': 'anthropic'},
    {'company': 'Scale AI', 'ats': 'greenhouse', 'board_slug': 'scaleai'},

    # Lever (3) — 현재 확인 가능한 기술기업 보드가 상대적으로 적어 표본 비중이 작음
    {'company': 'Palantir', 'ats': 'lever', 'board_slug': 'palantir'},
    {'company': 'Spotify', 'ats': 'lever', 'board_slug': 'spotify'},
    {'company': 'Binance', 'ats': 'lever', 'board_slug': 'binance'},

    # Ashby (15)
    {'company': 'OpenAI', 'ats': 'ashby', 'board_slug': 'openai'},
    {'company': 'Notion', 'ats': 'ashby', 'board_slug': 'notion'},
    {'company': 'Ramp', 'ats': 'ashby', 'board_slug': 'ramp'},
    {'company': 'Cursor', 'ats': 'ashby', 'board_slug': 'cursor'},
    {'company': 'Linear', 'ats': 'ashby', 'board_slug': 'linear'},
    {'company': 'Perplexity', 'ats': 'ashby', 'board_slug': 'perplexity'},
    {'company': 'Replit', 'ats': 'ashby', 'board_slug': 'replit'},
    {'company': 'PostHog', 'ats': 'ashby', 'board_slug': 'posthog'},
    {'company': 'Supabase', 'ats': 'ashby', 'board_slug': 'supabase'},
    {'company': 'Modal', 'ats': 'ashby', 'board_slug': 'modal'},
    {'company': 'Cohere', 'ats': 'ashby', 'board_slug': 'cohere'},
    {'company': 'ElevenLabs', 'ats': 'ashby', 'board_slug': 'elevenlabs'},
    {'company': 'LangChain', 'ats': 'ashby', 'board_slug': 'langchain'},
    {'company': 'Render', 'ats': 'ashby', 'board_slug': 'render'},
    {'company': 'Railway', 'ats': 'ashby', 'board_slug': 'railway'},
]

companies_df = pd.DataFrame(COMPANIES)
assert len(companies_df) == 30
display(companies_df.groupby('ats').size().rename('companies').reset_index())
display(companies_df)

,ats,companies
0,ashby,15
1,greenhouse,12
2,lever,3


,company,ats,board_slug
0,Stripe,greenhouse,stripe
1,Cloudflare,greenhouse,cloudflare
2,Datadog,greenhouse,datadog
3,Discord,greenhouse,discord
4,Figma,greenhouse,figma
5,Cockroach Labs,greenhouse,cockroachlabs
6,MongoDB,greenhouse,mongodb
7,Lyft,greenhouse,lyft
8,Roblox,greenhouse,roblox
9,Airbnb,greenhouse,airbnb


## 2. 공개 API 호출

각 회사당 요청은 한 번입니다. 일시적인 429·502·503·504에만 짧게 재시도하며, 실패한 회사를 조용히 누락하지 않고 상태표에 남깁니다.

In [3]:
def build_url(ats, board_slug):
    if ats == 'greenhouse':
        return f'https://boards-api.greenhouse.io/v1/boards/{board_slug}/jobs?content=true'
    if ats == 'lever':
        return f'https://api.lever.co/v0/postings/{board_slug}?mode=json'
    if ats == 'ashby':
        return f'https://api.ashbyhq.com/posting-api/job-board/{board_slug}?includeCompensation=true'
    raise ValueError(f'지원하지 않는 ATS: {ats}')

def extract_job_list(ats, payload):
    if ats in {'greenhouse', 'ashby'}:
        return payload.get('jobs', [])
    if ats == 'lever':
        return payload if isinstance(payload, list) else []
    return []

def fetch_company(spec, max_attempts=3):
    url = build_url(spec['ats'], spec['board_slug'])
    started = time.monotonic()
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url,
                headers={'Accept': 'application/json', 'User-Agent': USER_AGENT},
                timeout=REQUEST_TIMEOUT,
            )
            if response.status_code in {429, 502, 503, 504} and attempt < max_attempts:
                time.sleep(2 ** (attempt - 1))
                continue
            response.raise_for_status()
            payload = response.json()
            jobs = extract_job_list(spec['ats'], payload)
            return {
                **spec, 'ok': True, 'http_status': response.status_code,
                'job_count': len(jobs), 'elapsed_sec': round(time.monotonic() - started, 2),
                'error': None, 'url': url, 'payload': payload,
            }
        except (requests.RequestException, ValueError) as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            if attempt < max_attempts:
                time.sleep(2 ** (attempt - 1))

    return {
        **spec, 'ok': False, 'http_status': None, 'job_count': 0,
        'elapsed_sec': round(time.monotonic() - started, 2),
        'error': last_error, 'url': url, 'payload': None,
    }

In [4]:
fetch_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(fetch_company, spec) for spec in COMPANIES]
    for future in as_completed(futures):
        fetch_results.append(future.result())

fetch_results.sort(key=lambda x: (x['ats'], x['company']))
raw_payloads = {(r['ats'], r['board_slug']): r['payload'] for r in fetch_results if r['ok']}

fetch_status_df = pd.DataFrame([
    {k: r[k] for k in ['company', 'ats', 'board_slug', 'ok', 'http_status',
                          'job_count', 'elapsed_sec', 'error', 'url']}
    for r in fetch_results
])

print(f"성공 회사: {int(fetch_status_df['ok'].sum())}/{len(fetch_status_df)}")
print('전체 원본 공고:', int(fetch_status_df['job_count'].sum()))
display(fetch_status_df)

성공 회사: 30/30
전체 원본 공고: 5791


,company,ats,board_slug,ok,http_status,job_count,elapsed_sec,error,url
0,Cohere,ashby,cohere,True,200,143,0.35,None,https://api.ashbyhq.com/posting-api/job-board/...
1,Cursor,ashby,cursor,True,200,118,0.31,None,https://api.ashbyhq.com/posting-api/job-board/...
2,ElevenLabs,ashby,elevenlabs,True,200,240,0.70,None,https://api.ashbyhq.com/posting-api/job-board/...
3,LangChain,ashby,langchain,True,200,101,0.30,None,https://api.ashbyhq.com/posting-api/job-board/...
4,Linear,ashby,linear,True,200,31,0.28,None,https://api.ashbyhq.com/posting-api/job-board/...
5,Modal,ashby,modal,True,200,31,0.30,None,https://api.ashbyhq.com/posting-api/job-board/...
6,Notion,ashby,notion,True,200,128,0.32,None,https://api.ashbyhq.com/posting-api/job-board/...
7,OpenAI,ashby,openai,True,200,731,0.61,None,https://api.ashbyhq.com/posting-api/job-board/...
8,Perplexity,ashby,perplexity,True,200,93,0.30,None,https://api.ashbyhq.com/posting-api/job-board/...
9,PostHog,ashby,posthog,True,200,11,0.28,None,https://api.ashbyhq.com/posting-api/job-board/...


## 3. ATS별 응답을 공통 스키마로 정규화

In [5]:
def html_to_text(value):
    if not value:
        return ''
    text = unescape(str(value))
    text = re.sub(r'(?i)<br\s*/?>|</p>|</li>|</h[1-6]>', '\n', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = unescape(text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()

def join_nonempty(values):
    seen, result = set(), []
    for value in values:
        value = str(value or '').strip()
        if value and value not in seen:
            seen.add(value)
            result.append(value)
    return '\n\n'.join(result)

def normalize_greenhouse(spec, job):
    description_html = job.get('content') or ''
    return {
        'source': 'greenhouse', 'company': spec['company'], 'board_slug': spec['board_slug'],
        'source_job_id': str(job.get('id', '')), 'title': job.get('title'),
        'description_html': description_html, 'description': html_to_text(description_html),
        'location': (job.get('location') or {}).get('name'),
        'department': ', '.join(x.get('name', '') for x in job.get('departments', []) if x.get('name')),
        'team': None, 'employment_type': None,
        'published_at': job.get('first_published'), 'updated_at': job.get('updated_at'),
        'source_url': job.get('absolute_url'), 'apply_url': job.get('absolute_url'),
        'collected_at': COLLECTED_AT,
    }

def normalize_lever(spec, job):
    list_parts = [html_to_text(x.get('content')) for x in job.get('lists', [])]
    description = join_nonempty([
        job.get('descriptionPlain'), job.get('openingPlain'), *list_parts, job.get('additionalPlain')
    ])
    categories = job.get('categories') or {}
    created_at = job.get('createdAt')
    published_at = (
        datetime.fromtimestamp(created_at / 1000, tz=timezone.utc).isoformat()
        if isinstance(created_at, (int, float)) else None
    )
    return {
        'source': 'lever', 'company': spec['company'], 'board_slug': spec['board_slug'],
        'source_job_id': str(job.get('id', '')), 'title': job.get('text'),
        'description_html': job.get('description') or '', 'description': description,
        'location': categories.get('location'), 'department': categories.get('department'),
        'team': categories.get('team'), 'employment_type': categories.get('commitment'),
        'published_at': published_at, 'updated_at': None,
        'source_url': job.get('hostedUrl'), 'apply_url': job.get('applyUrl'),
        'collected_at': COLLECTED_AT,
    }

def normalize_ashby(spec, job):
    return {
        'source': 'ashby', 'company': spec['company'], 'board_slug': spec['board_slug'],
        'source_job_id': str(job.get('id', '')), 'title': job.get('title'),
        'description_html': job.get('descriptionHtml') or '',
        'description': job.get('descriptionPlain') or html_to_text(job.get('descriptionHtml')),
        'location': job.get('location'), 'department': job.get('department'), 'team': job.get('team'),
        'employment_type': job.get('employmentType'), 'published_at': job.get('publishedAt'),
        'updated_at': None, 'source_url': job.get('jobUrl'), 'apply_url': job.get('applyUrl'),
        'collected_at': COLLECTED_AT,
    }

NORMALIZERS = {
    'greenhouse': normalize_greenhouse,
    'lever': normalize_lever,
    'ashby': normalize_ashby,
}

In [6]:
normalized_jobs = []
for result in fetch_results:
    if not result['ok']:
        continue
    spec = {k: result[k] for k in ['company', 'ats', 'board_slug']}
    for raw_job in extract_job_list(result['ats'], result['payload']):
        normalized_jobs.append(NORMALIZERS[result['ats']](spec, raw_job))

jobs_df = pd.DataFrame(normalized_jobs)
jobs_df['description'] = jobs_df['description'].fillna('').astype(str)
jobs_df['description_length'] = jobs_df['description'].str.len()
jobs_df['job_key'] = jobs_df['source'] + ':' + jobs_df['board_slug'] + ':' + jobs_df['source_job_id']

print('정규화 공고 수:', len(jobs_df))
display(jobs_df.drop(columns=['description_html', 'description']).head(10))

정규화 공고 수: 5791


,source,company,board_slug,source_job_id,title,location,department,team,employment_type,published_at,updated_at,source_url,apply_url,collected_at,description_length,job_key
0,ashby,Cohere,cohere,cd3eacfe-1169-4df0-8164-93a857d5ddf0,Revenue Operations Analyst (Analytics),New York,Business Operations,Finance,FullTime,2026-05-31T04:10:26.913+00:00,NaN,https://jobs.ashbyhq.com/cohere/cd3eacfe-1169-...,https://jobs.ashbyhq.com/cohere/cd3eacfe-1169-...,2026-08-11T13:48:14.537617+00:00,5694,ashby:cohere:cd3eacfe-1169-4df0-8164-93a857d5ddf0
1,ashby,Cohere,cohere,31c47498-0ccf-4d23-a418-0d2c616ba909,"Member of Technical Staff, MLE (Korea)",Korea,Modeling,Applied-ML,FullTime,2024-10-31T10:59:40.461+00:00,NaN,https://jobs.ashbyhq.com/cohere/31c47498-0ccf-...,https://jobs.ashbyhq.com/cohere/31c47498-0ccf-...,2026-08-11T13:48:14.537617+00:00,5296,ashby:cohere:31c47498-0ccf-4d23-a418-0d2c616ba909
2,ashby,Cohere,cohere,3136a5a5-06fd-4c82-8b72-a43467e6b128,"Member of Technical Staff, Modeling",London,Modeling,Modeling,FullTime,2024-11-01T16:06:43.318+00:00,NaN,https://jobs.ashbyhq.com/cohere/3136a5a5-06fd-...,https://jobs.ashbyhq.com/cohere/3136a5a5-06fd-...,2026-08-11T13:48:14.537617+00:00,5179,ashby:cohere:3136a5a5-06fd-4c82-8b72-a43467e6b128
3,ashby,Cohere,cohere,443368a3-6276-4b90-9671-27fed40fd6d2,"Senior Member of Technical Staff, Multimodal AI",San Francisco,Modeling,Modeling,FullTime,2024-12-03T14:25:43.571+00:00,NaN,https://jobs.ashbyhq.com/cohere/443368a3-6276-...,https://jobs.ashbyhq.com/cohere/443368a3-6276-...,2026-08-11T13:48:14.537617+00:00,6653,ashby:cohere:443368a3-6276-4b90-9671-27fed40fd6d2
4,ashby,Cohere,cohere,acec6038-117b-400b-92e3-0745fbb4cf53,Data Annotation Specialist - German Writer/Tra...,Canada,Data Quality (Contract),Multilingual Annotation - German,Contract,2026-02-20T21:09:49.620+00:00,NaN,https://jobs.ashbyhq.com/cohere/acec6038-117b-...,https://jobs.ashbyhq.com/cohere/acec6038-117b-...,2026-08-11T13:48:14.537617+00:00,5919,ashby:cohere:acec6038-117b-400b-92e3-0745fbb4cf53
5,ashby,Cohere,cohere,b9c8c98e-b0fa-43b6-93b0-fa780d956066,"Software Engineer, Security",Toronto,Product,Security,FullTime,2026-01-27T19:39:04.733+00:00,NaN,https://jobs.ashbyhq.com/cohere/b9c8c98e-b0fa-...,https://jobs.ashbyhq.com/cohere/b9c8c98e-b0fa-...,2026-08-11T13:48:14.537617+00:00,5327,ashby:cohere:b9c8c98e-b0fa-43b6-93b0-fa780d956066
6,ashby,Cohere,cohere,d42f5fd4-1ffc-45b9-957c-f09862db6af6,"Member of Technical Staff, Training Performanc...",London,Modeling,Modeling,FullTime,2025-02-20T10:45:59.057+00:00,NaN,https://jobs.ashbyhq.com/cohere/d42f5fd4-1ffc-...,https://jobs.ashbyhq.com/cohere/d42f5fd4-1ffc-...,2026-08-11T13:48:14.537617+00:00,5228,ashby:cohere:d42f5fd4-1ffc-45b9-957c-f09862db6af6
7,ashby,Cohere,cohere,a13207e7-dc82-473f-8ca4-e832452fe8c3,"Member of Technical Staff, Training Infra Engi...",Paris,Modeling,Modeling,FullTime,2025-02-20T15:53:36.664+00:00,NaN,https://jobs.ashbyhq.com/cohere/a13207e7-dc82-...,https://jobs.ashbyhq.com/cohere/a13207e7-dc82-...,2026-08-11T13:48:14.537617+00:00,5048,ashby:cohere:a13207e7-dc82-473f-8ca4-e832452fe8c3
8,ashby,Cohere,cohere,6d0f0753-ff22-46fd-90e9-08998914a8e7,Solutions Architect - Public Sector,"Washington, DC",Revenue,Solutions Architecture,FullTime,2026-03-19T17:24:07.254+00:00,NaN,https://jobs.ashbyhq.com/cohere/6d0f0753-ff22-...,https://jobs.ashbyhq.com/cohere/6d0f0753-ff22-...,2026-08-11T13:48:14.537617+00:00,6717,ashby:cohere:6d0f0753-ff22-46fd-90e9-08998914a8e7
9,ashby,Cohere,cohere,a912a36f-0300-41d3-997f-636cc85a5e1b,Data Annotation Specialist - Investment Bankin...,Canada,Data Quality (Contract),Finance Annotation,Contract,2026-07-30T19:09:57.979+00:00,NaN,https://jobs.ashbyhq.com/cohere/a912a36f-0300-...,https://jobs.ashbyhq.com/cohere/a912a36f-0300-...,2026-08-11T13:48:14.537617+00:00,5262,ashby:cohere:a912a36f-0300-41d3-997f-636cc85a5e1b


In [7]:
# 분석용 최소 컬럼 DataFrame
jobs_text_df = jobs_df[['company', 'title', 'description']].copy()

print(f'전체 공고 수: {len(jobs_text_df):,}')
display(jobs_text_df.head(1000))

전체 공고 수: 5,791


,company,title,description
0,Cohere,Revenue Operations Analyst (Analytics),Who are we?\n\nCohere is the leading security-...
1,Cohere,"Member of Technical Staff, MLE (Korea)",Who are we?\n\nCohere is the leading security-...
2,Cohere,"Member of Technical Staff, Modeling",Who are we?\n\nCohere is the leading security-...
3,Cohere,"Senior Member of Technical Staff, Multimodal AI",Who are we?\n\nCohere is the leading security-...
4,Cohere,Data Annotation Specialist - German Writer/Tra...,Who are we?\n\nOur mission is to scale intelli...
...,...,...,...
995,OpenAI,"Client Partner, Ads Solutions",About the team\n\nOpenAI’s mission is to ensur...
996,OpenAI,"Applied AI Architect, Large Enterprise",About the Team\n\nThe AI Architect team partne...
997,OpenAI,"Backend Engineer, Consumer Devices",ABOUT THE TEAM\n\nThe Software Engineering tea...
998,OpenAI,"Frontend Engineer, Financial Web Platform",About the team\n\nThe Applied team at OpenAI s...


## title -> 직군 추출

**title -> 정규화 -> Bigram TF-IDF -> LinearSVC(or LogisticReg) -> job_role**

1. 직군 taxonomy 확정
        

2. title 정규화
        

3. 일부 title에 job_role 라벨링
        

4. train / validation 분리
        

5. Bigram TF-IDF
        

6. LinearSVC 학습
        

7. F1 / confusion matrix 검증
        
        
8. 전체 5,754건에 job_role 생성

In [8]:
# 1. 직군 taxonomy (분류체계)

JOB_ROLES = [
    "General Software Engineer",
    "Backend / Server",
    "Frontend",
    "Full-stack",
    "Mobile",
    "Data Engineer / Data Platform",
    "Data Analyst",
    "ML / AI Engineer / Data Scientist",
    "Research Engineer / Scientist",
    "Infra / Platform / DevOps / SRE",
    "Systems / Low-level",
    "Security Engineer",
    "QA / Test Engineering",
    "Solutions / Forward Deployed",
    "Engineering Management",
    "Excluded / AI Data Operations",
    "Other / Non-Developer",
]

In [9]:
# 2. title 정규화

## jobs_text_df가 company, title, description으로 구성됨

import re
import pandas as pd

'''
Senior Backend Engineer - Payments
→ senior backend engineer payments

Machine Learning Engineer (Inference)
→ machine learning engineer

Staff Data Engineer | Platform
→ staff data engineer platform
'''

def normalize_title(title):
    if pd.isna(title):
        return ""

    title = str(title)

    # 소문자화
    title = title.lower()

    # 특수 구분자 통일
    title = re.sub(r"[_/|]+", " ", title)
    title = re.sub(r"[-–—]+", " ", title)

    # 괄호 안 내용 제거
    title = re.sub(r"\([^)]*\)", " ", title)
    title = re.sub(r"\[[^\]]*\]", " ", title)

    # 불필요한 공백 제거
    title = re.sub(r"\s+", " ", title).strip()

    return title


jobs_text_df["normalized_title"] = (
    jobs_text_df["title"]
    .apply(normalize_title)
)

display(
    jobs_text_df[
        ["title", "normalized_title"]
    ].head(8)
)

,title,normalized_title
0,Revenue Operations Analyst (Analytics),revenue operations analyst
1,"Member of Technical Staff, MLE (Korea)","member of technical staff, mle"
2,"Member of Technical Staff, Modeling","member of technical staff, modeling"
3,"Senior Member of Technical Staff, Multimodal AI","senior member of technical staff, multimodal ai"
4,Data Annotation Specialist - German Writer/Tra...,data annotation specialist german writer trans...
5,"Software Engineer, Security","software engineer, security"
6,"Member of Technical Staff, Training Performanc...","member of technical staff, training performanc..."
7,"Member of Technical Staff, Training Infra Engi...","member of technical staff, training infra engi..."


In [10]:
# 3. 학습용 title 만들기

## 어쨌든 지도학습이니 train data (label)이 필요함
## title 중복 제거해서 unique title 목록 제작
title_label_df = (
    jobs_text_df[
        ["title", "normalized_title"]
    ]
    .drop_duplicates(subset=["normalized_title"])
    .reset_index(drop=True)
)

print(f"전체 공고 수: {len(jobs_text_df):,}")
print(f"Unique title 수: {len(title_label_df):,}")

display(title_label_df.head(10))

전체 공고 수: 5,791
Unique title 수: 4,625


,title,normalized_title
0,Revenue Operations Analyst (Analytics),revenue operations analyst
1,"Member of Technical Staff, MLE (Korea)","member of technical staff, mle"
2,"Member of Technical Staff, Modeling","member of technical staff, modeling"
3,"Senior Member of Technical Staff, Multimodal AI","senior member of technical staff, multimodal ai"
4,Data Annotation Specialist - German Writer/Tra...,data annotation specialist german writer trans...
5,"Software Engineer, Security","software engineer, security"
6,"Member of Technical Staff, Training Performanc...","member of technical staff, training performanc..."
7,"Member of Technical Staff, Training Infra Engi...","member of technical staff, training infra engi..."
8,Solutions Architect - Public Sector,solutions architect public sector
9,Data Annotation Specialist - Investment Bankin...,data annotation specialist investment banking ...


In [11]:
# ============================================================
# 3. 학습/검증용 title 500개 샘플링
#    - 개발/기술직 후보 400개
#    - 비개발직 후보 100개
# ============================================================

import re
import pandas as pd


# ------------------------------------------------------------
# 개발/기술직 "후보"를 넓게 잡는 패턴
# → 여기서는 최종 직군 분류가 아니라 샘플링 용도임
# ------------------------------------------------------------
TECH_CANDIDATE_PATTERN = re.compile(
    r"""
    \b(
        engineer |
        engineering |
        developer |
        software |
        backend |
        frontend |
        full[\s-]?stack |
        mobile |
        ios |
        android |
        data |
        machine[\s-]?learning |
        ml |
        ai |
        artificial[\s-]?intelligence |
        scientist |
        research |
        sre |
        devops |
        platform |
        infrastructure |
        security |
        server   |
        systems? |
        compiler |
        runtime |
        firmware |
        kernel |
        architect |
        technical[\s-]?staff |
        member[\s-]?of[\s-]?technical[\s-]?staff
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)


# unique title 기준
title_label_df["is_tech_candidate"] = (
    title_label_df["normalized_title"]
    .fillna("")
    .apply(lambda x: bool(TECH_CANDIDATE_PATTERN.search(x)))
)


tech_pool = title_label_df[
    title_label_df["is_tech_candidate"]
].copy()

nontech_pool = title_label_df[
    ~title_label_df["is_tech_candidate"]
].copy()


# 기술직 400 + 비기술직 100
tech_sample = tech_pool.sample(
    n=min(400, len(tech_pool)),
    random_state=42
)

nontech_sample = nontech_pool.sample(
    n=min(100, len(nontech_pool)),
    random_state=42
)


label_sample_df = (
    pd.concat(
        [tech_sample, nontech_sample],
        ignore_index=True
    )
    .sample(frac=1, random_state=42)  # 순서 섞기
    .reset_index(drop=True)
)


# 최종 정답 컬럼
label_sample_df["job_role"] = ""


print(f"기술직 후보 pool: {len(tech_pool):,}")
print(f"비기술직 pool   : {len(nontech_pool):,}")
print(f"최종 sample     : {len(label_sample_df):,}")

print(
    "\n샘플 구성:",
    label_sample_df["is_tech_candidate"]
    .value_counts()
    .rename(index={True: "TECH", False: "NON-TECH"})
)

display(
    label_sample_df[
        ["title", "normalized_title", "is_tech_candidate", "job_role"]
    ].head(30)
)

기술직 후보 pool: 2,297
비기술직 pool   : 2,328
최종 sample     : 500

샘플 구성: is_tech_candidate
TECH        400
NON-TECH    100
Name: count, dtype: int64


,title,normalized_title,is_tech_candidate,job_role
0,"Engineering Manager, Home Infrastructure","engineering manager, home infrastructure",True,
1,Academic Research Partnerships & Programs Lead,academic research partnerships & programs lead,True,
2,"ML Software Engineer, ETA","ml software engineer, eta",True,
3,Enterprise Sales Engineer - Rockies,enterprise sales engineer rockies,True,
4,"Staff+ Software Engineer, Data Infrastructure","staff+ software engineer, data infrastructure",True,
5,"Senior Software Engineer, Marketplace","senior software engineer, marketplace",True,
6,"Staff+ Software Engineer, Financial Fraud","staff+ software engineer, financial fraud",True,
7,Actuator Electromagnetic Design Engineer,actuator electromagnetic design engineer,True,
8,"Manager II, Technical Escalations Engineering","manager ii, technical escalations engineering",True,
9,"Account Manager, Strategic Healthcare Partners...","account manager, strategic healthcare partners...",False,


In [ ]:
# ============================================================
# 4. CSV keyword 기반 Weak Labeling
#    1) Other / Non-Developer 정식 클래스 추가
#    2) boundary-aware matching
#    3) 긴 phrase 우선
# ============================================================

import re
import pandas as pd


# ------------------------------------------------------------
# 1. CSV 로드
# ------------------------------------------------------------
role_keyword_df = pd.read_csv("role_title_keywords.csv")

required_cols = {"role_category", "title_keyword"}

if not required_cols.issubset(role_keyword_df.columns):
    raise ValueError(
        f"CSV에 {required_cols} 컬럼이 필요합니다. "
        f"현재 컬럼: {role_keyword_df.columns.tolist()}"
    )

# 기존 CSV taxonomy를 현재 분류체계로 보정한다.
role_keyword_df["role_category"] = role_keyword_df["role_category"].replace({
    "ML / AI Engineer": "ML / AI Engineer / Data Scientist",
    "ML / AI Engineering": "ML / AI Engineer / Data Scientist",
    "Data Scientist / Analyst": "Data Analyst",
    "Data Science / Analyst": "Data Analyst",
})


# ------------------------------------------------------------
# 2. 비개발 직군 규칙 추가
# ------------------------------------------------------------
NON_DEV_KEYWORDS = [
    # Account / Sales
    "Account Executive",
    "Account Manager",
    "Account Director",
    "Strategic Account",
    "Enterprise Account",
    "Sales Manager",
    "Sales Director",
    "Sales Representative",
    "Sales Development Representative",
    "Business Development",

    # HR / Recruiting / People
    "HR",
    "Human Resources",
    "HR Business Partner",
    "People Operations",
    "People Partner",
    "People Manager",
    "Talent Acquisition",
    "Talent Partner",
    "Recruiter",
    "Recruiting",

    # Product / Program
    "Product Manager",
    "Product Management",
    "Program Manager",
    "Technical Program Manager",

    # Customer
    "Customer Success",
    "Customer Success Manager",
    "Customer Support",
    "Support Specialist",

    # Design
    "Product Designer",
    "UX Designer",
    "UI Designer",
    "Graphic Designer",
    "Design Manager",

    # Marketing
    "Marketing Manager",
    "Marketing Specialist",
    "Product Marketing",
    "Growth Marketing",

    # Finance / Legal
    "Finance Manager",
    "Financial Analyst",
    "Accounting",
    "Accountant",
    "Legal Counsel",
    "General Counsel",

    # Operations / Admin
    "Operations Manager",
    "Business Operations",
    "Revenue Operations",
    "Office Manager",
    "Executive Assistant",
    "Payroll Specialist",

    # 확실한 비개발/인접 기술 직군
    "Sales Engineer",
    "Sales Engineering",
    "Pre-Sales",
    "Technical Support Engineer",
    "Technical Services Engineer",
    "Mechanical Engineer",
    "Mechanical Design Engineer",
    "Electrical Engineer",
    "Finance Systems",
    "Business Systems",
    "Research Operations",
    "Engineering Compensation",

    # empty_roles.csv 기반 추가 제외 규칙
    "Seller Systems Operations",
    "Systems Operations Associate",
    "Signal Integrity Engineer",
    "PCBA Manufacturing Engineer",
    "Manufacturing Engineer",
    "Corporate IT Engineer",
    "IT Support Engineer",
    "National Security Policy",
    "National Security Hackathon",
    "General Interest",
    "Academic Research Partnerships",
    "Research Partnerships",
    "Consulting Engineer",
    "Premier Support Engineering",
    "Support Engineering",
    "Support Engineer",
    "Technical Services Curriculum",
    "Senior Design Engineer",
    "Product Quality Engineer",
    "Component and Product Quality Engineer",
    "Data Center Design Engineer",
    "Data Center Architect",
    "Data Center Energy",
    "Data Center Compute",
    "GTM Architect",
    "GTM Innovation",
    "Technical Sourcer",
    "Audio Engineering Lead",
    "FP&A Analyst",
    "Product Counsel",
    "Partner Operations",
    "Operations & Systems",
    "Professional Services Engineer",
    "Hardware Operations Engineer",
    "Regional Security Manager",
    "Head of International Security",
    "CX Knowledge Architect",
    "Design Systems Lead",
    "Automation Engineer - Influencers",
    "Outcomes Architect",
    "Oracle Technical Architect",
]


non_dev_df = pd.DataFrame({
    "role_category": "Other / Non-Developer",
    "title_keyword": NON_DEV_KEYWORDS
})

role_keyword_df = pd.concat(
    [role_keyword_df, non_dev_df],
    ignore_index=True
)


# ------------------------------------------------------------
# 3. 핵심 기술직 keyword 보강
# ------------------------------------------------------------
# 짧은 keyword도 아래 boundary matcher를 쓰므로
# retail 속 ai 같은 오탐은 발생하지 않음.

ADDITIONAL_TECH_RULES = [
    # AI / ML
    ("ML / AI Engineer / Data Scientist", "AI"),
    ("ML / AI Engineer / Data Scientist", "ML"),
    ("ML / AI Engineer / Data Scientist", "MLE"),
    ("ML / AI Engineer / Data Scientist", "mle"),
    ("ML / AI Engineer / Data Scientist", "Machine Learning"),
    ("ML / AI Engineer / Data Scientist", "Artificial Intelligence"),
    ("ML / AI Engineer / Data Scientist", "Algorithm Engineer"),
    ("ML / AI Engineer / Data Scientist", "Model Behavior Engineer"),
    ("ML / AI Engineer / Data Scientist", "Training Performance Engineer"),

    # Mobile
    ("Mobile", "iOS"),
    ("Mobile", "Android"),

    # General SWE
    ("General Software Engineer", "Software Engineer"),
    ("General Software Engineer", "Software Developer"),
    ("General Software Engineer", "Member of Technical Staff"),

    # Data Engineering
    ("Data Engineer / Data Platform", "Data Engineer"),
    ("Data Engineer / Data Platform", "Data Platform"),
    ("Data Engineer / Data Platform", "Analytics Engineer"),

    # ML / AI / Data Scientist
    ("ML / AI Engineer / Data Scientist", "Data Scientist"),
    ("ML / AI Engineer / Data Scientist", "Data Science"),
    ("ML / AI Engineer / Data Scientist", "Decision Scientist"),
    ("ML / AI Engineer / Data Scientist", "Applied Scientist"),

    # Data Analyst
    ("Data Analyst", "Data Analyst"),
    ("Data Analyst", "Product Analyst"),
    ("Data Analyst", "Analytics Analyst"),
    ("Data Analyst", "Business Intelligence Analyst"),
    ("Data Analyst", "BI Analyst"),

    # Research
    ("Research Engineer / Scientist", "Research Engineer"),
    ("Research Engineer / Scientist", "Research Scientist"),

    # Infra / Platform / SRE
    ("Infra / Platform / DevOps / SRE", "Site Reliability Engineer"),
    ("Infra / Platform / DevOps / SRE", "SRE"),
    ("Infra / Platform / DevOps / SRE", "DevOps Engineer"),
    ("Infra / Platform / DevOps / SRE", "Infrastructure Engineer"),
    ("Infra / Platform / DevOps / SRE", "Platform Engineer"),
    ("Infra / Platform / DevOps / SRE", "Network Deployment Engineer"),
    ("Infra / Platform / DevOps / SRE", "Network Deployment"),
    ("Infra / Platform / DevOps / SRE", "Network Operations Engineer"),
    ("Infra / Platform / DevOps / SRE", "Network Operations"),
    ("Infra / Platform / DevOps / SRE", "Optical Network Engineer"),
    ("Infra / Platform / DevOps / SRE", "Optical Network"),
    ("Infra / Platform / DevOps / SRE", "Deployment Engineer"),
    ("Infra / Platform / DevOps / SRE", "Reliability Engineer"),

    # Systems / Low-level
    ("Systems / Low-level", "Distributed Systems Engineer"),
    ("Systems / Low-level", "Systems Engineer"),
    ("Systems / Low-level", "Firmware Engineer"),
    ("Systems / Low-level", "Firmware"),
    ("Systems / Low-level", "Silicon Engineer"),
    ("Systems / Low-level", "Silicon"),
    ("Systems / Low-level", "Compiler Engineer"),
    ("Systems / Low-level", "Compiler"),
    ("Systems / Low-level", "Runtime Engineer"),
    ("Systems / Low-level", "Runtime"),
    ("Systems / Low-level", "Performance Engineer"),

    # Security
    ("Security Engineer", "Security Engineer"),
    ("Security Engineer", "Detection & Mitigation"),
    ("Security Engineer", "GRC Engineer"),
    ("Security Engineer", "Vulnerability Management"),
    ("Security Engineer", "Threat Intel"),
    ("Security Engineer", "Threat Investigator"),
    ("Security Engineer", "Security Analyst"),
    ("Security Engineer", "Security Architect"),
    ("Security Engineer", "Code Security"),

    # Solutions
    ("Solutions / Forward Deployed", "Solutions Engineer"),
    ("Solutions / Forward Deployed", "Forward Deployed Engineer"),

    # Research / ML
    ("Research Engineer / Scientist", "Agent Post-Training Research"),
    ("Research Engineer / Scientist", "Post-Training Research"),

    # Security
    ("Security Engineer", "Red Team Engineer"),
    ("Security Engineer", "Red Team"),

    # Infra / Platform / SRE
    ("Infra / Platform / DevOps / SRE", "Cloud Operations Engineer"),
    ("Infra / Platform / DevOps / SRE", "Cloud Operations"),

    # Solutions / Forward Deployed
    ("Solutions / Forward Deployed", "Solution Architect"),
    ("Solutions / Forward Deployed", "Partner Solution Architect"),
    ("Solutions / Forward Deployed", "Specialist Solution Architect"),

    # Engineering Management
    ("Engineering Management", "Engineering Manager"),
    ("Engineering Management", "Director of Engineering"),
    ("Engineering Management", "Director, Engineering"),
    ("Engineering Management", "Manager, Software Engineering"),
    ("Engineering Management", "Manager Software Engineering"),
    ("Engineering Management", "Manager I, Engineering"),
    ("Engineering Management", "Manager II, Engineering"),
    ("Engineering Management", "Manager, Engineering"),
    ("Engineering Management", "Technical Lead Manager"),
    ("Engineering Management", "Regional Director, Forward Deployed Engineering"),
]


additional_tech_df = pd.DataFrame(
    ADDITIONAL_TECH_RULES,
    columns=["role_category", "title_keyword"]
)

role_keyword_df = pd.concat(
    [role_keyword_df, additional_tech_df],
    ignore_index=True
)


# 중복 제거
role_keyword_df = (
    role_keyword_df
    .drop_duplicates(
        subset=["role_category", "title_keyword"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. 매칭용 정규화
# ------------------------------------------------------------
def normalize_for_matching(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # punctuation을 공백으로
    text = re.sub(
        r"[-_/|()\[\],:;]+",
        " ",
        text
    )

    # 다중 공백 제거
    text = re.sub(r"\s+", " ", text).strip()

    return text


role_keyword_df["keyword_norm"] = (
    role_keyword_df["title_keyword"]
    .apply(normalize_for_matching)
)


# ------------------------------------------------------------
# 5. 긴 phrase를 우선
# ------------------------------------------------------------
role_keyword_df["keyword_length"] = (
    role_keyword_df["keyword_norm"]
    .str.len()
)

role_keyword_df = (
    role_keyword_df
    .sort_values(
        "keyword_length",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. boundary-aware phrase matcher
# ------------------------------------------------------------
def contains_keyword(title, keyword):
    """
    단순 substring 대신 단어 경계를 검사.

    예:
    AI      ↛ retail
    API     ↛ capital
    ML      ↛ html

    하지만:
    AI Engineer
    ML Engineer
    iOS Engineer
    에는 정상 매칭.
    """

    title = normalize_for_matching(title)
    keyword = normalize_for_matching(keyword)

    pattern = (
        rf"(?<![a-zA-Z0-9])"
        rf"{re.escape(keyword)}"
        rf"(?![a-zA-Z0-9])"
    )

    return bool(
        re.search(
            pattern,
            title,
            flags=re.IGNORECASE
        )
    )


# ------------------------------------------------------------
# 7. 직군 weak labeling
# ------------------------------------------------------------
def assign_job_role(title):

    title_norm = normalize_for_matching(title)

    # -----------------------------------------
    # 강제 우선 규칙
    # -----------------------------------------

    # 확실한 비개발/인접 기술 직군은 개발직 규칙보다 우선한다.
    for keyword in NON_DEV_KEYWORDS:
        if contains_keyword(title_norm, keyword):
            return "Other / Non-Developer"

    # Mobile은 굉장히 명확
    if contains_keyword(title_norm, "ios"):
        return "Mobile"

    if contains_keyword(title_norm, "android"):
        return "Mobile"

    # Data Scientist는 ML/AI 계열과 합친다.
    if (
        contains_keyword(title_norm, "data scientist")
        or contains_keyword(title_norm, "data science")
        or contains_keyword(title_norm, "decision scientist")
        or contains_keyword(title_norm, "applied scientist")
    ):
        return "ML / AI Engineer / Data Scientist"

    # Data Analyst는 Data Scientist와 분리한다.
    if (
        contains_keyword(title_norm, "data analyst")
        or contains_keyword(title_norm, "product analyst")
        or contains_keyword(title_norm, "analytics analyst")
        or contains_keyword(title_norm, "business intelligence analyst")
        or contains_keyword(title_norm, "bi analyst")
    ):
        return "Data Analyst"

    # AI / ML은 사용자 정의상 ML/AI/Data Scientist로 우선
    if (
        contains_keyword(title_norm, "ai")
        or contains_keyword(title_norm, "ml")
        or contains_keyword(title_norm, "mle")
    ):
        return "ML / AI Engineer / Data Scientist"

    # -----------------------------------------
    # CSV 규칙
    # -----------------------------------------
    for _, row in role_keyword_df.iterrows():

        if contains_keyword(
            title_norm,
            row["keyword_norm"]
        ):
            return row["role_category"]

    # -----------------------------------------
    # 매칭되지 않은 title은 NaN으로 남긴 뒤 학습에서 제외한다.
    # -----------------------------------------
    return pd.NA


# ------------------------------------------------------------
# 8. 500개 샘플에 적용
# ------------------------------------------------------------
label_sample_df["job_role"] = (
    label_sample_df["title"]
    .apply(assign_job_role)
)


# keyword에 안 잡힌 NON-TECH 표본은
# Other / Non-Developer로 넣는다.
nontech_unmatched_mask = (
    (~label_sample_df["is_tech_candidate"])
    &
    (
        label_sample_df["job_role"].isna()
        | (label_sample_df["job_role"] == "")
    )
)

label_sample_df.loc[
    nontech_unmatched_mask,
    "job_role"
] = "Other / Non-Developer"


# ------------------------------------------------------------
# 9. 결과 확인
# ------------------------------------------------------------
print("=" * 60)
print("직군별 초벌 라벨")
print("=" * 60)

job_role_for_count = (
    label_sample_df["job_role"]
    .replace("", pd.NA)
    .fillna("UNMATCHED")
)

print(job_role_for_count.value_counts())

unmatched_count = (
    label_sample_df["job_role"]
    .replace("", pd.NA)
    .isna()
    .sum()
)

print(
    f"\nUNMATCHED: "
    f"{unmatched_count}/{len(label_sample_df)} "
    f"({unmatched_count / len(label_sample_df):.1%})"
)


print("\n[UNMATCHED title]")
display(
    label_sample_df.loc[
        label_sample_df["job_role"].replace("", pd.NA).isna(),
        [
            "title",
            "normalized_title",
            "is_tech_candidate",
            "job_role"
        ]
    ].head(100)
)

직군별 초벌 라벨
job_role
Other / Non-Developer                159
General Software Engineer             89
ML / AI Engineer / Data Scientist     60
UNMATCHED                             52
Engineering Management                36
Solutions / Forward Deployed          28
Security Engineer                     14
Research Engineer / Scientist         14
Infra / Platform / DevOps / SRE       13
Systems / Low-level                   11
Backend / Server                       7
Data Engineer / Data Platform          4
Mobile                                 4
Frontend                               3
Full-stack                             3
Data Analyst                           2
Data Annotation / AI Ops성 직무           1
Name: count, dtype: int64

UNMATCHED: 52/500 (10.4%)

[UNMATCHED title]


,title,normalized_title,is_tech_candidate,job_role
7,Actuator Electromagnetic Design Engineer,actuator electromagnetic design engineer,True,NaN
8,"Manager II, Technical Escalations Engineering","manager ii, technical escalations engineering",True,NaN
18,Senior Infra Engineer: Observability,senior infra engineer: observability,True,NaN
37,"Manager, Security Operations","manager, security operations",True,NaN
39,Engineer 3 - GTM Tech,engineer 3 gtm tech,True,NaN
40,"Technical Enablement Lead, Claude Platform","technical enablement lead, claude platform",True,NaN
41,Manufacturing Test Engineer,manufacturing test engineer,True,NaN
50,"Product Engineer, Product Platform","product engineer, product platform",True,NaN
61,"Technical Operations Engineer, Payments","technical operations engineer, payments",True,NaN
73,Object Storage Product Engineer,object storage product engineer,True,NaN


In [13]:
# UNMATCHED는 따로 저장하고, 학습용 데이터에서는 제거한다.
label_sample_df["job_role"] = label_sample_df["job_role"].replace("", pd.NA)

empty_df = label_sample_df[label_sample_df["job_role"].isna()].copy()
empty_df.to_csv("empty_roles.csv", index=False)

label_sample_df = (
    label_sample_df
    .dropna(subset=["job_role"])
    .reset_index(drop=True)
)

print(f"드랍된 UNMATCHED title: {len(empty_df):,}")
print(f"학습용 labeled title: {len(label_sample_df):,}")

드랍된 UNMATCHED title: 52
학습용 labeled title: 448


In [14]:
## 제작된 데이터 기반으로 학습 수행

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2), ## unigram
            lowercase=True,
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced"
        )
    )
])

In [15]:
# ============================================================
# 학습/검증 성능 확인
# - NaN job_role 제거
# - 제외 클래스 제거
# - 샘플 수 2개 미만 클래스 제거
# - stratify 가능한 데이터만 train/validation split
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

EXCLUDE_ROLES = {
    "Excluded / AI Data Operations",
    "Data Annotation / AI Ops성 직무",
}

train_df = label_sample_df.copy()

# 빈 라벨 / NaN 제거
train_df["job_role"] = train_df["job_role"].replace("", pd.NA)
train_df = train_df.dropna(subset=["job_role"]).copy()

# 학습에서 제외할 클래스 제거
train_df = train_df[
    ~train_df["job_role"].isin(EXCLUDE_ROLES)
].copy()

# 클래스별 샘플 수 확인
role_counts = train_df["job_role"].value_counts()

print("학습 후보 클래스 분포")
print(role_counts)

# stratify를 위해 샘플 수 2개 미만 클래스 제거
too_small_roles = role_counts[role_counts < 2].index.tolist()

if too_small_roles:
    print("\n샘플 수가 2개 미만이라 학습/검증에서 제외할 클래스:")
    print(too_small_roles)

train_df = train_df[
    ~train_df["job_role"].isin(too_small_roles)
].copy()

print("\n최종 학습 클래스 분포")
print(train_df["job_role"].value_counts())

X = train_df["normalized_title"]
y = train_df["job_role"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

print("\nValidation classification report")
print(
    classification_report(
        y_val,
        y_pred,
        zero_division=0
    )
)

학습 후보 클래스 분포
job_role
Other / Non-Developer                159
General Software Engineer             89
ML / AI Engineer / Data Scientist     60
Engineering Management                36
Solutions / Forward Deployed          28
Security Engineer                     14
Research Engineer / Scientist         14
Infra / Platform / DevOps / SRE       13
Systems / Low-level                   11
Backend / Server                       7
Data Engineer / Data Platform          4
Mobile                                 4
Frontend                               3
Full-stack                             3
Data Analyst                           2
Name: count, dtype: int64

최종 학습 클래스 분포
job_role
Other / Non-Developer                159
General Software Engineer             89
ML / AI Engineer / Data Scientist     60
Engineering Management                36
Solutions / Forward Deployed          28
Security Engineer                     14
Research Engineer / Scientist         14
Infra / Platform / DevOps /

In [16]:
# ============================================================
# 오분류 샘플 확인
# ============================================================

error_df = pd.DataFrame({
    "normalized_title": X_val,
    "true_role": y_val,
    "pred_role": y_pred
})

error_df = error_df[
    error_df["true_role"] != error_df["pred_role"]
].copy()

print(f"오분류 수: {len(error_df)}")
display(error_df.head(100))

오분류 수: 10


,normalized_title,true_role,pred_role
189,"mobile engineer, treasury",Mobile,Backend / Server
219,"senior machine learning engineer, public sector",ML / AI Engineer / Data Scientist,Other / Non-Developer
57,binance accelerator program brand kol,Other / Non-Developer,Mobile
111,"backend software engineer, gtm innovation",Other / Non-Developer,General Software Engineer
286,"engineering manager, community support enginee...",Other / Non-Developer,Engineering Management
198,"member of technical staff, senior staff mle",ML / AI Engineer / Data Scientist,General Software Engineer
31,senior staff product engineer,Engineering Management,Other / Non-Developer
299,engineering compensation partner,Other / Non-Developer,Engineering Management
358,customer service representative,Other / Non-Developer,Solutions / Forward Deployed
93,"machine learning engineer, distributed data sy...",ML / AI Engineer / Data Scientist,Systems / Low-level


In [17]:
# ============================================================
# 3. 전체 unique title에 job_role 예측
# ============================================================

# 전체 unique title 기준
all_title_df = title_label_df.copy()

all_title_df["pred_job_role"] = model.predict(
    all_title_df["normalized_title"]
)

print("전체 unique title 예측 분포")
print(all_title_df["pred_job_role"].value_counts())

display(all_title_df.head(30))

전체 unique title 예측 분포
pred_job_role
Other / Non-Developer                2531
General Software Engineer             630
ML / AI Engineer / Data Scientist     353
Solutions / Forward Deployed          248
Engineering Management                237
Security Engineer                     117
Research Engineer / Scientist         115
Backend / Server                      104
Infra / Platform / DevOps / SRE        98
Systems / Low-level                    56
Data Analyst                           48
Mobile                                 39
Full-stack                             24
Data Engineer / Data Platform          17
Frontend                                8
Name: count, dtype: int64


,title,normalized_title,is_tech_candidate,pred_job_role
0,Revenue Operations Analyst (Analytics),revenue operations analyst,False,Other / Non-Developer
1,"Member of Technical Staff, MLE (Korea)","member of technical staff, mle",True,General Software Engineer
2,"Member of Technical Staff, Modeling","member of technical staff, modeling",True,General Software Engineer
3,"Senior Member of Technical Staff, Multimodal AI","senior member of technical staff, multimodal ai",True,ML / AI Engineer / Data Scientist
4,Data Annotation Specialist - German Writer/Tra...,data annotation specialist german writer trans...,True,ML / AI Engineer / Data Scientist
5,"Software Engineer, Security","software engineer, security",True,General Software Engineer
6,"Member of Technical Staff, Training Performanc...","member of technical staff, training performanc...",True,General Software Engineer
7,"Member of Technical Staff, Training Infra Engi...","member of technical staff, training infra engi...",True,General Software Engineer
8,Solutions Architect - Public Sector,solutions architect public sector,True,Solutions / Forward Deployed
9,Data Annotation Specialist - Investment Bankin...,data annotation specialist investment banking ...,True,Other / Non-Developer


In [18]:
# ============================================================
# 4. 전체 공고 데이터에 job_role 붙이기
# ============================================================

jobs_with_role_df = jobs_text_df.merge(
    all_title_df[["normalized_title", "pred_job_role"]],
    on="normalized_title",
    how="left"
)

jobs_with_role_df = jobs_with_role_df.rename(
    columns={"pred_job_role": "job_role"}
)

print(f"전체 공고 수: {len(jobs_with_role_df):,}")
print(jobs_with_role_df["job_role"].value_counts())

display(
    jobs_with_role_df[
        ["company", "title", "normalized_title", "job_role", "description"]
    ].head(30)
)

전체 공고 수: 5,791
job_role
Other / Non-Developer                3178
General Software Engineer             848
ML / AI Engineer / Data Scientist     404
Solutions / Forward Deployed          344
Engineering Management                265
Security Engineer                     151
Research Engineer / Scientist         140
Backend / Server                      121
Infra / Platform / DevOps / SRE       120
Systems / Low-level                    59
Data Analyst                           53
Mobile                                 47
Full-stack                             32
Data Engineer / Data Platform          20
Frontend                                9
Name: count, dtype: int64


,company,title,normalized_title,job_role,description
0,Cohere,Revenue Operations Analyst (Analytics),revenue operations analyst,Other / Non-Developer,Who are we?\n\nCohere is the leading security-...
1,Cohere,"Member of Technical Staff, MLE (Korea)","member of technical staff, mle",General Software Engineer,Who are we?\n\nCohere is the leading security-...
2,Cohere,"Member of Technical Staff, Modeling","member of technical staff, modeling",General Software Engineer,Who are we?\n\nCohere is the leading security-...
3,Cohere,"Senior Member of Technical Staff, Multimodal AI","senior member of technical staff, multimodal ai",ML / AI Engineer / Data Scientist,Who are we?\n\nCohere is the leading security-...
4,Cohere,Data Annotation Specialist - German Writer/Tra...,data annotation specialist german writer trans...,ML / AI Engineer / Data Scientist,Who are we?\n\nOur mission is to scale intelli...
5,Cohere,"Software Engineer, Security","software engineer, security",General Software Engineer,Who are we?\n\nCohere is the leading security-...
6,Cohere,"Member of Technical Staff, Training Performanc...","member of technical staff, training performanc...",General Software Engineer,Who are we?\n\nCohere is the leading security-...
7,Cohere,"Member of Technical Staff, Training Infra Engi...","member of technical staff, training infra engi...",General Software Engineer,Who are we?\n\nCohere is the leading security-...
8,Cohere,Solutions Architect - Public Sector,solutions architect public sector,Solutions / Forward Deployed,Who are we?\n\nCohere is the leading security-...
9,Cohere,Data Annotation Specialist - Investment Bankin...,data annotation specialist investment banking ...,Other / Non-Developer,Who are we?\n\nCohere is the leading security-...


In [19]:
# ============================================================
# 5. 개발직 분석용 데이터셋 생성
# ============================================================

EXCLUDE_ROLES = {
    "Other / Non-Developer",
    "Excluded / AI Data Operations",
}

dev_role_jobs_df = jobs_with_role_df[
    ~jobs_with_role_df["job_role"].isin(EXCLUDE_ROLES)
].copy()

print(f"전체 공고 수: {len(jobs_with_role_df):,}")
print(f"개발/기술직 분석 대상: {len(dev_role_jobs_df):,}")

print(dev_role_jobs_df["job_role"].value_counts())

display(
    dev_role_jobs_df[
        ["company", "title", "job_role", "description"]
    ].head(30)
)

전체 공고 수: 5,791
개발/기술직 분석 대상: 2,613
job_role
General Software Engineer            848
ML / AI Engineer / Data Scientist    404
Solutions / Forward Deployed         344
Engineering Management               265
Security Engineer                    151
Research Engineer / Scientist        140
Backend / Server                     121
Infra / Platform / DevOps / SRE      120
Systems / Low-level                   59
Data Analyst                          53
Mobile                                47
Full-stack                            32
Data Engineer / Data Platform         20
Frontend                               9
Name: count, dtype: int64


,company,title,job_role,description
1,Cohere,"Member of Technical Staff, MLE (Korea)",General Software Engineer,Who are we?\n\nCohere is the leading security-...
2,Cohere,"Member of Technical Staff, Modeling",General Software Engineer,Who are we?\n\nCohere is the leading security-...
3,Cohere,"Senior Member of Technical Staff, Multimodal AI",ML / AI Engineer / Data Scientist,Who are we?\n\nCohere is the leading security-...
4,Cohere,Data Annotation Specialist - German Writer/Tra...,ML / AI Engineer / Data Scientist,Who are we?\n\nOur mission is to scale intelli...
5,Cohere,"Software Engineer, Security",General Software Engineer,Who are we?\n\nCohere is the leading security-...
6,Cohere,"Member of Technical Staff, Training Performanc...",General Software Engineer,Who are we?\n\nCohere is the leading security-...
7,Cohere,"Member of Technical Staff, Training Infra Engi...",General Software Engineer,Who are we?\n\nCohere is the leading security-...
8,Cohere,Solutions Architect - Public Sector,Solutions / Forward Deployed,Who are we?\n\nCohere is the leading security-...
10,Cohere,"Member of Technical Staff, Search",General Software Engineer,Who are we?\n\nCohere is the leading security-...
11,Cohere,"Member of Technical Staff, Post-Training",General Software Engineer,Who are we?\n\nCohere is the leading security-...


In [22]:
dev_role_jobs_df.to_csv('dev_role_jobs_tagged.csv', index=False)